# 实机诊断包复算（2026-09-24）

数据为手持绕操场行走的记录，不能用作静止 IMU 噪声、零偏或 GPS 漂移标定。
起飞总质量 734 g、GPS M1025 来自操作者说明；完整结论与下一轮步骤见同目录 README.md。

依赖：`mcap`、`mcap-ros2-support`、`numpy`、`pyyaml`。在本仓库内启动 notebook。
直接读取 MCAP 内嵌 schema，校验存储 CRC 和消息数量；不回放到 ROS，不连接任何串口。
分析算法保存在同目录 `analyze.py`。所有 GPS accuracy 数值都是接收机报告，非真值误差。


In [1]:
from pathlib import Path
import importlib.util
import json
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'agi_ros2/config/hardware.yaml').is_file())
folder = root / 'agi_ros2/analysis/hardware_diagnostic_20260924'
bag = root / 'bags/hardware_20260924_150053_980611'
spec = importlib.util.spec_from_file_location('diagnostic_analysis', folder / 'analyze.py')
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
result = analysis.summarize(bag)
saved = json.loads((folder / 'summary.json').read_text())
assert result == saved, 'Recomputed evidence differs from saved summary'
print(json.dumps(result['validation'], ensure_ascii=False, indent=2))


{
  "reader": "mcap.NonSeekingReader with embedded ROS 2 schemas",
  "validate_crcs": true,
  "crc_scope": "stored chunk/data CRCs, where present",
  "metadata_counts_match": true,
  "messages": 375359,
  "metadata_messages": 375359,
  "topic_count": 27,
  "duration_seconds": 224.01361429500002
}


In [2]:
for field, unit in [('horizontal_accuracy', 'm'), ('vertical_accuracy', 'm'),
                    ('velocity_accuracy', 'm/s')]:
    stats = result['measurements']['gps_observations/' + field]
    print(f"{field}: median={stats['median']:.3f}, p95={stats['p95']:.3f}, "
          f"maximum={stats['maximum']:.3f} {unit}, finite={stats['finite']}")
print('GPS observation / revocation counts:',
      result['streams']['gps_observations']['messages'],
      result['streams']['navigation_revocations']['messages'])
print('Heading validity:', result['counts']['/sensors/fc_heading/valid'])
print('Fusion initialized:', result['counts']['/fused_state/initialized'])
print('Raw profiles:', result['counts']['msp_status/pid_profile'],
      result['counts']['msp_status/rate_profile'])
print('Readbacks:')
for entry in result['msp_readbacks']:
    if entry['code'] in (1, 2, 3, 111, 12304):
        print(entry['code'], entry['decoded'])


horizontal_accuracy: median=1.969, p95=3.077, maximum=3.670 m, finite=2180
vertical_accuracy: median=3.466, p95=4.348, maximum=4.999 m, finite=2180
velocity_accuracy: median=0.813, p95=1.586, maximum=2.267 m/s, finite=2180
GPS observation / revocation counts: 2180 2180
Heading validity: Counter({'false': 2180})
Fusion initialized: Counter({'false': 110579})
Raw profiles: Counter({'0': 5533}) Counter({'0': 5533})
Readbacks:
1 {'protocol': 0, 'api': '1.48'}
2 {'variant': 'BTFL'}
3 {'version_bytes': [26, 6, 1], 'payload_length': 12, 'version_string': '2026.6.1'}
111 {'rates_type': 3, 'throttle_limit_type': 0, 'center_rate_deg_s': [70, 70, 70], 'max_rate_deg_s': [670, 670, 670], 'expo_percent': [0, 0, 0], 'rate_limits_deg_s': [1998, 1998, 1998]}
12304 {'request_name': 'msp_override_channels_mask', 'reply': 'msp_override_channels_mask = 0'}
12304 {'request_name': 'msp_override_failsafe', 'reply': 'msp_override_failsafe = OFF'}
12304 {'request_name': 'msp_override_timeout_ms', 'reply': 'msp_

## 用修复后的解码器检查原始 MSP 证据

这一步只调用 Python 解码函数，使用包内请求时间与会话，不向 FC 发送数据。
它应接受合法的 1.48 / 2026.6.1 版本回复，但仍因真实的 mask=0 拒绝授权。


In [3]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import yaml
sys.path.insert(0, str(root / 'agi_ros2/scripts'))
from shadow_support import MspEvidence

config = yaml.safe_load((root / 'agi_ros2/config/hardware.yaml').read_text())
expected = dict(config['bridge'])
expected.update({key: value for key, value in config['evidence'].items()
                 if key not in ('geofence_min', 'geofence_max', 'battery_timeout')})
decoder = MspEvidence(expected)
with next(bag.glob('*.mcap')).open('rb') as source:
    reader = make_reader(source, decoder_factories=[DecoderFactory()], validate_crcs=True)
    for _, _, record, message in reader.iter_decoded_messages(topics=['/msp/events']):
        timestamp = message.request_stamp.sec + message.request_stamp.nanosec * 1e-9
        decoder.accept(message.code, message.payload, timestamp, message.session_id,
                       message.request_name, message.event)
        now = record.log_time * 1e-9
snapshot = decoder.snapshot(now)
print({key: snapshot[key] for key in ('config_verified', 'reason', 'override_timeout_ms')})
assert snapshot['reason'] == 'Override mask must be 15 (AETR only)'
assert not snapshot['config_verified']


{'config_verified': False, 'reason': 'Override mask must be 15 (AETR only)', 'override_timeout_ms': 300}
